# Lesson 04 - Modelul de Proiectare Tool Use

În această lecție veți învăța modelul de proiectare **Tool Use** pentru agenții AI folosind Microsoft Agent Framework (Python). Vom acoperi:

- Definirea uneltelor funcționale cu decoratorul `@tool` și parametri tipizați
- Furnizarea schemelor uneltelor pentru ca modelul să știe ce face fiecare unealtă
- Controlul execuției uneltei cu `approval_mode`
- Returnarea de **output structurat** prin modelele Pydantic și `response_format`

Scenariul este un **agent de rezervare călătorii** care poate căuta destinații, verifica disponibilitatea și recupera informații despre zboruri.


## Configurare


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Definirea Uneltelor cu Decoratorul @tool

Decoratorul `@tool` transformă o funcție Python simplă într-o unealtă pe care un agent o poate apela.  
Puncte cheie:

- **docstring** devine descrierea uneltei pe care modelul o vede.  
- **Anotările de tip** (inclusiv `Annotated` cu descrieri) definesc schema uneltei.  
- `approval_mode` controlează dacă utilizatorul trebuie să aprobe fiecare apel înainte de execuție.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Crearea unui agent cu mai multe unelte

Trimite toate cele trei unelte către client astfel încât modelul să poată apela oricare dintre ele are nevoie pentru a răspunde la întrebarea utilizatorului.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Ieșire structurată cu unelte

Prin setarea `response_format` la un model Pydantic, agentul este forțat să returneze un obiect JSON bine tipizat în loc de text liber. Acest lucru este util când codul ulterior trebuie să consume rezultatul programatic.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Modele de aprobare a instrumentului

Parametrul `approval_mode` la `@tool` controlează dacă apelurile instrumentului necesită aprobare umană înainte de execuție:

| Mod | Comportament |
|---|---|
| `"never_require"` | Instrumentul rulează automat — nu este nevoie de confirmare din partea utilizatorului. |
| `"always_require"` | Fiecare apel trebuie aprobat de utilizator înainte de a fi executat. |

Folosește `"always_require"` pentru instrumente care au efecte secundare (de exemplu, rezervarea unui zbor, încărcarea cardului de credit) astfel încât un om să rămână în buclă.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

În această lecție ai învățat cum să:

1. **Definești instrumente** folosind decoratorul `@tool` cu parametri tipați și docstrings care servesc drept schema instrumentului.
2. **Compozi mai multe instrumente** astfel încât agentul să le poată apela în secvență pentru a răspunde la interogări complexe.
3. **Returnezi un output structurat** prin transmiterea unui model Pydantic ca `response_format`.
4. **Controlezi aprobarea instrumentelor** cu `approval_mode` pentru a menține un om implicat în proces pentru operațiuni sensibile.

Aceste tipare formează baza pentru construirea de agenți fiabili, pregătiți pentru producție, care pot interacționa în siguranță cu sisteme externe.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
